# 03 — Clean invalid records and flag suspicious prices
 **Purpose:** remove invalid prices, exclude Borong/low-coverage items and preserve suspicious prices for review.
 **Saved:** one daily Parquet, one training price-review CSV and one cleaning summary containing the learned bounds.
 A price flag is evidence to inspect, not proof of an error. The default retains flagged prices.

Up to six fixed reviewed records are conservatively excluded in the holdout training period. Expanding folds retain them by default because review dates were not recorded. Other statistical flags are retained. These suspected errors are not externally confirmed; optional Notebook 90 checks sensitivity to keeping them. No manual review CSV is required.


In [ ]:
import pandas as pd
import numpy as np
import json
import hashlib
import pricecatcher_config as cfg
from importlib import reload
cfg = reload(cfg)
print("Experiment:", cfg.EXPERIMENT, "| Training cutoff:", cfg.TRAIN_END_DATE)

preparation = json.loads((cfg.STAGE2_DIR / "preparation_settings.json").read_text())
expected = {**cfg.PREPARATION_SETTINGS, "source_manifest_sha256": hashlib.sha256(
    (cfg.STAGE1_DIR / "source_manifest.json").read_bytes()).hexdigest()}
if preparation != expected:
    raise ValueError("Preparation settings/source changed. Rerun 02–05 for this experiment.")
df = pd.read_parquet(cfg.STAGE1_COMBINED_PARQUET)
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["price"] = pd.to_numeric(df["price"], errors="coerce")
if df["date"].isna().any():
    raise ValueError("Missing source dates: rerun/check 01.")
df = df.loc[df["date"].between(pd.Timestamp(cfg.START_MONTH + "-01"),
                                  pd.Timestamp(cfg.EVALUATION_END_DATE))].copy()
original_rows = len(df)
required = ["date", "premise_code"] + cfg.GROUP_KEYS
if df[required].isna().any().any():
    raise ValueError("Missing dates or grouping details: inspect the integration output.")
if df.duplicated(["date", "premise_code", "item_code"]).any():
    raise ValueError("Duplicate price keys need review before cleaning.")

# Small source-level tables keep fully excluded places/series visible in later audits.
geography = ["state", "district"]
daily_geography = df.groupby(geography).size().rename("source_records").to_frame()
source_series = df.groupby(cfg.GROUP_KEYS).agg(
    source_records=("price", "size"), source_observed_days=("date", "nunique"))
source_series.to_csv(cfg.STAGE3_DIR / "source_series_coverage.csv")


## 1. Remove invalid prices and apply item/premise exclusions


In [ ]:
invalid = df["price"].isna() | ~np.isfinite(df["price"]) | (df["price"] <= 0)
invalid_count = int(invalid.sum())
daily_geography["invalid_removed"] = df.loc[invalid].groupby(geography).size()
invalid_review = df.loc[invalid & (df["date"] <= pd.Timestamp(cfg.TRAIN_END_DATE))].copy()
invalid_review["review_reason"] = "invalid price"
df = df.loc[~invalid].copy()
print("Invalid prices removed:", invalid_count)

coverage = pd.read_csv(cfg.STAGE2_ITEM_COVERAGE_CSV)
if not coverage["training_end"].astype(str).eq(cfg.TRAIN_END_DATE).all():
    raise ValueError("Coverage report uses a different training cutoff. Rerun Notebook 02.")
low_codes = coverage.loc[coverage["exclude_low_coverage"].eq(True), "item_code"].tolist()
borong = df["premise_type"].eq("Borong") if cfg.REMOVE_BORONG else pd.Series(False, index=df.index)
low_coverage = df["item_code"].isin(low_codes) & ~borong
borong_count, low_count = int(borong.sum()), int(low_coverage.sum())
daily_geography["borong_removed"] = df.loc[borong].groupby(geography).size()
daily_geography["low_coverage_item_removed"] = df.loc[low_coverage].groupby(geography).size()
model = df.loc[~borong & ~low_coverage].copy()
if model.empty:
    raise ValueError("No records remain after item/premise exclusions.")
print("Borong excluded:", borong_count, "| Low-coverage records excluded:", low_count)


## 2. Learn price bounds from training retail data
 Bounds use the valid training basket before low-coverage exclusions, preserving the earlier flagging rule. No validation/test prices determine these bounds.


In [ ]:
reference = df[df["date"] <= pd.Timestamp(cfg.TRAIN_END_DATE)].copy()
if cfg.REMOVE_BORONG:
    reference = reference[reference["premise_type"] != "Borong"]
if reference.empty:
    raise ValueError("No training data available for price bounds.")

stats = reference.groupby("item_code")["price"].agg(
    q1=lambda x: x.quantile(0.25), median="median", q3=lambda x: x.quantile(0.75))
stats["iqr"] = stats["q3"] - stats["q1"]
stats["lower_bound"] = pd.concat([stats["q1"] - cfg.IQR_MULTIPLIER * stats["iqr"],
                                  stats["median"] * cfg.LOW_MEDIAN_MULTIPLIER], axis=1).max(axis=1).clip(lower=0)
stats["upper_bound"] = pd.concat([stats["q3"] + cfg.IQR_MULTIPLIER * stats["iqr"],
                                  stats["median"] * cfg.HIGH_MEDIAN_MULTIPLIER], axis=1).min(axis=1)
zero_iqr = stats["iqr"] == 0
stats.loc[zero_iqr, "lower_bound"] = stats.loc[zero_iqr, "median"] * cfg.LOW_MEDIAN_MULTIPLIER
stats.loc[zero_iqr, "upper_bound"] = stats.loc[zero_iqr, "median"] * cfg.HIGH_MEDIAN_MULTIPLIER

model = model.merge(stats[["lower_bound", "upper_bound"]], on="item_code", how="left", validate="many_to_one")
model["is_suspicious_price"] = (model["price"] < model["lower_bound"]) | (model["price"] > model["upper_bound"])
print("Flagged modelling records:", int(model["is_suspicious_price"].sum()))
print("Records without training bounds:", int(model["lower_bound"].isna().sum()))


## 2a. Apply training-dated documented conservative exclusions
TIMUN: four RM55 records at premise 19117 follow RM5.50 observations in the same calendar-week pair. Sugar: two RM28.50 records replace RM2.85 while other matched premises remain at RM2.85. Exact tenfold patterns support conservative exclusion, not an invented correction. Bounds above remain fixed so the sensitivity comparison isolates the six-record decision. Other flagged values, including extreme high values, remain unverified and retained; report this residual uncertainty.

Exception discovery was retrospective; fold date checks prevent evaluation-dated exclusions but do not prove real-time error detection.


In [ ]:
reviewed_records = pd.DataFrame([
    ["2025-02-27", 19117, 113, 55.0],
    ["2025-02-28", 19117, 113, 55.0],
    ["2025-03-01", 19117, 113, 55.0],
    ["2025-03-02", 19117, 113, 55.0],
    ["2025-09-29", 1291, 1589, 28.5],
    ["2025-09-29", 20872, 1589, 28.5],
], columns=["date", "premise_code", "item_code", "expected_price"])
reviewed_records["date"] = pd.to_datetime(reviewed_records["date"])
reviewed_records["reason"] = "Suspected tenfold source-price issue; conservative exclusion, not confirmed error"
# Do not use an evaluation-period observation to decide a specific source exception.
# These fixed retrospective exceptions remain a research limitation, not confirmed errors.
reviewed_records = reviewed_records.loc[
    reviewed_records["date"].le(pd.Timestamp(cfg.TRAIN_END_DATE))
    & ~reviewed_records["item_code"].isin(low_codes)].copy()
if not cfg.APPLY_DOCUMENTED_EXCLUSIONS:
    reviewed_records = reviewed_records.iloc[0:0].copy()
keys = ["date", "premise_code", "item_code"]
matched = reviewed_records.merge(model, on=keys, how="left", validate="one_to_one")
if matched["price"].isna().any() or not np.isclose(matched["price"], matched["expected_price"]).all():
    raise ValueError("Reviewed records differ from this source snapshot. Check before excluding.")
reviewed_daily = matched[cfg.FINAL_COLS + ["is_suspicious_price"]].copy()
daily_geography["documented_removed"] = reviewed_daily.groupby(geography).size()
model = model.merge(reviewed_records[keys].assign(exclude=True), on=keys, how="left", validate="one_to_one")
model = model.loc[model["exclude"].ne(True)].drop(columns="exclude")
display(reviewed_records)
print("Documented records excluded:", len(reviewed_records))


In [ ]:
reviewed_records.to_csv(cfg.STAGE3_DIR / "documented_exclusions.csv", index=False)


## 3. Save daily data and review evidence
Statistical flags are retained by default. The source-review CSV is an evidence export, not a file you must edit. Keep the original integrated dataset.


In [ ]:
flagged_review = model.loc[model["is_suspicious_price"] & (model["date"] <= pd.Timestamp(cfg.TRAIN_END_DATE))].copy()
flagged_review["review_reason"] = "outside training price bounds; cause unconfirmed"
review = pd.concat([invalid_review, flagged_review], ignore_index=True)
review.to_csv(cfg.STAGE3_DIR / "training_price_review.csv", index=False)
print("Training records for source review:", len(review))
display(review[cfg.FINAL_COLS + ["review_reason"]].head(10))

flagged_count = int(model["is_suspicious_price"].sum())
flagged_removed = flagged_count if cfg.REMOVE_FLAGGED_PRICES else 0
daily_geography["statistical_flag_removed"] = (
    model.loc[model["is_suspicious_price"]].groupby(geography).size()
    if cfg.REMOVE_FLAGGED_PRICES else 0)
if cfg.REMOVE_FLAGGED_PRICES:
    model = model.loc[~model["is_suspicious_price"]].copy()
model = model[cfg.FINAL_COLS + ["is_suspicious_price"]]
if model.empty:
    raise ValueError("No records remain after price screening.")
model.to_parquet(cfg.STAGE3_MODEL_READY_PARQUET, index=False)
daily_geography["retained_records"] = model.groupby(geography).size()
daily_geography = daily_geography.fillna(0).astype(int)
assert daily_geography.drop(columns="source_records").sum(axis=1).eq(daily_geography["source_records"]).all()
daily_geography.to_csv(cfg.STAGE3_DIR / "daily_geographical_coverage.csv")
summary = {"preparation": preparation,
    "reviewed_records_excluded": len(reviewed_records),
    "target_status": cfg.TARGET_STATUS,
    "reviewed_daily_records": reviewed_daily.assign(date=reviewed_daily["date"].dt.strftime("%Y-%m-%d")).to_dict(orient="records"),
    "review_policy": {"documented_training_records_excluded": len(reviewed_records),
                      "statistical_flags_removed": cfg.REMOVE_FLAGGED_PRICES,
                      "replacement_prices": False},"original_rows": original_rows, "invalid_removed": invalid_count,
    "borong_removed": borong_count, "low_coverage_removed": low_count,
    "excluded_item_codes": [int(c) for c in low_codes],
    "flagged_records_before_optional_removal": flagged_count,
    "remove_flagged_prices": cfg.REMOVE_FLAGGED_PRICES,
    "flagged_removed": flagged_removed,
    "daily_rows_saved": len(model), "item_codes": int(model["item_code"].nunique()),
    "training_end": cfg.TRAIN_END_DATE,
    "flag_rule": {"iqr_multiplier": cfg.IQR_MULTIPLIER,
                  "low_median_multiplier": cfg.LOW_MEDIAN_MULTIPLIER,
                  "high_median_multiplier": cfg.HIGH_MEDIAN_MULTIPLIER},
    "training_price_bounds": stats.reset_index().to_dict(orient="records")}
(cfg.STAGE3_DIR / "cleaning_summary.json").write_text(json.dumps(summary, indent=2))
print("Saved daily rows:", len(model))
print("Next: Notebook 04.")
